# E-Commerce Recommendation Engine

**Prerequisites:** Run the setup script first to create the ML environment:
```bash
chmod +x setup_ml_environment.sh
./setup_ml_environment.sh
```
Then select **Kernel → Change Kernel → 'Python (ML Env)'** in Jupyter.

## Phase 0: Setup & Libraries

In [1]:
# ========================================
# CELL 1: ENVIRONMENT VERIFICATION
# ========================================
# This cell verifies you're using the correct kernel

import sys
import os

# Check if using virtual environment
venv_path = os.environ.get('VIRTUAL_ENV', '')
if 'ml_env' in sys.executable or 'ml_env' in venv_path:
    print("✓ Using ML virtual environment")
    print(f"  Python: {sys.executable}")
else:
    print("⚠ WARNING: Not using the ML virtual environment!")
    print(f"  Current Python: {sys.executable}")
    print("")
    print("  To fix this:")
    print("  1. Go to Kernel → Change Kernel → 'Python (ML Env)'")
    print("  2. Or run: ~/ml_env/bin/jupyter notebook")

✓ Using ML virtual environment
  Python: /home/jami/ml_env/bin/python


✓ All libraries imported successfully!


In [3]:
# ========================================
# CELL 3: VERIFY GPU & LIBRARY VERSIONS
# ========================================

print("=" * 60)
print("LIBRARY STATUS")
print("=" * 60)

# Core libraries
print(f"\n📦 Core Libraries:")
print(f"   numpy:        {np.__version__}")
print(f"   pandas:       {pd.__version__}")
print(f"   matplotlib:   {plt.matplotlib.__version__}")
print(f"   seaborn:      {sns.__version__}")

# ML libraries
import sklearn
import surprise
print(f"\n🤖 ML Libraries:")
print(f"   scikit-learn: {sklearn.__version__}")
print(f"   surprise:     {surprise.__version__}")

# Deep Learning
import transformers
print(f"\n🧠 Deep Learning:")
print(f"   torch:        {torch.__version__}")
print(f"   transformers: {transformers.__version__}")

# GPU Status
print(f"\n🎮 GPU STATUS:")
if torch.cuda.is_available():
    print(f"   CUDA Available:  ✓ Yes")
    print(f"   CUDA Version:    {torch.version.cuda}")
    print(f"   GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory:      {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    
    # Quick GPU test
    test_tensor = torch.tensor([1.0, 2.0, 3.0]).cuda()
    print(f"   GPU Test:        ✓ Tensor operations working")
else:
    print(f"   CUDA Available:  ✗ No")
    print(f"   Training will use CPU (slower)")

print("\n" + "=" * 60)
print("✓ Environment ready for training!")
print("=" * 60)

LIBRARY STATUS

📦 Core Libraries:
   numpy:        1.26.4
   pandas:       2.3.3
   matplotlib:   3.10.8
   seaborn:      0.13.2

🤖 ML Libraries:
   scikit-learn: 1.7.2
   surprise:     1.1.4

🧠 Deep Learning:
   torch:        2.6.0+cu124
   transformers: 4.57.6

🎮 GPU STATUS:
   CUDA Available:  ✓ Yes
   CUDA Version:    12.4
   GPU Device:      NVIDIA GeForce GTX 1660 SUPER
   GPU Memory:      5.6 GB
   GPU Test:        ✓ Tensor operations working

✓ Environment ready for training!


In [4]:
# ========================================
# CELL 4: SET DEVICE FOR TRAINING
# ========================================

# Automatically select best available device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    # Set memory management for better performance on 6GB cards
    torch.cuda.empty_cache()
    print("GPU memory cache cleared")

Using device: cuda
GPU: NVIDIA GeForce GTX 1660 SUPER
GPU memory cache cleared


## Phase 1: Data Loading & Exploration

Add your data loading code below...

In [ ]:
# Example: Load Amazon product reviews dataset from HuggingFace
# Uncomment and modify as needed

# dataset = load_dataset("amazon_polarity", split="train[:10000]")
# print(f"Loaded {len(dataset)} samples")
# print(dataset[0])

## Phase 2: Data Preprocessing

In [ ]:
# Add your preprocessing code here

## Phase 3: Exploratory Data Analysis

In [ ]:
# Example visualization
# fig, ax = plt.subplots(figsize=(10, 6))
# sns.histplot(data=df, x='rating', ax=ax)
# plt.title('Rating Distribution')
# plt.show()

## Phase 4: Content-Based Filtering

In [ ]:
# TF-IDF based content filtering example
# tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
# tfidf_matrix = tfidf.fit_transform(df['description'])
# cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

## Phase 5: Collaborative Filtering (Surprise Library)

In [ ]:
# Collaborative filtering with SVD
# reader = Reader(rating_scale=(1, 5))
# data = Dataset.load_from_df(df[['user_id', 'product_id', 'rating']], reader)
# trainset, testset = surprise_train_test_split(data, test_size=0.2)

# svd = SVD(n_factors=100, n_epochs=20, random_state=42)
# svd.fit(trainset)
# predictions = svd.test(testset)
# print(f"RMSE: {accuracy.rmse(predictions)}")

## Phase 6: Neural Collaborative Filtering (PyTorch + GPU)

In [ ]:
# Neural Collaborative Filtering Model
class NCF(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64, hidden_dims=[128, 64, 32]):
        super(NCF, self).__init__()
        
        # Embeddings
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        
        # MLP layers
        layers = []
        input_dim = embedding_dim * 2
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(input_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.2))
            input_dim = hidden_dim
        
        layers.append(nn.Linear(hidden_dims[-1], 1))
        self.mlp = nn.Sequential(*layers)
        
    def forward(self, user_ids, item_ids):
        user_emb = self.user_embedding(user_ids)
        item_emb = self.item_embedding(item_ids)
        x = torch.cat([user_emb, item_emb], dim=-1)
        return self.mlp(x).squeeze()

print("✓ NCF model class defined")

In [ ]:
# Example: Initialize and move model to GPU
# num_users = df['user_id'].nunique()
# num_items = df['product_id'].nunique()
# model = NCF(num_users, num_items).to(device)
# print(f"Model on: {next(model.parameters()).device}")

## Phase 7: Transformer-Based Recommendations

In [ ]:
# Load a pre-trained model for sentiment analysis on reviews
# This helps understand product quality from review text

# model_name = "distilbert-base-uncased-finetuned-sst-2-english"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
# print(f"Loaded {model_name} on {device}")

## Phase 8: Model Training

In [ ]:
# Training loop template
def train_model(model, train_loader, optimizer, criterion, device, epochs=3):
    model.train()
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")
        total_loss = 0
        
        progress_bar = tqdm(train_loader, desc=f"Training")
        
        for batch in progress_bar:
            # Move data to device
            batch = {k: v.to(device) for k, v in batch.items()}
            
            # Forward pass
            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss if hasattr(outputs, 'loss') else criterion(outputs, batch['labels'])
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / len(train_loader)
        print(f"Average loss: {avg_loss:.4f}")
    
    print("\n✓ Training complete!")
    return model

print("✓ Training function defined")

## Phase 9: Hybrid Recommendations

In [ ]:
# Combine content-based and collaborative filtering
def hybrid_recommend(user_id, content_scores, collab_scores, alpha=0.5):
    """
    Combine content-based and collaborative filtering scores.
    alpha: weight for content-based (1-alpha for collaborative)
    """
    hybrid_scores = alpha * content_scores + (1 - alpha) * collab_scores
    return hybrid_scores

print("✓ Hybrid recommendation function defined")

## Phase 10: Evaluation & Results

In [ ]:
# Evaluation metrics
def evaluate_recommendations(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    
    return {'rmse': rmse, 'mae': mae}

print("✓ Evaluation function defined")

In [ ]:
# Final GPU memory summary
if torch.cuda.is_available():
    print("GPU Memory Summary:")
    print(f"  Allocated: {torch.cuda.memory_allocated(0) / 1024**2:.1f} MB")
    print(f"  Cached:    {torch.cuda.memory_reserved(0) / 1024**2:.1f} MB")
    print(f"  Total:     {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")